# Day 11 Project Solution: Semantic Search Over Your Notes

Complete, working reference build. All five lesson concepts composed into one
pipeline: embed notes once at index time, embed the query at search time,
rank by cosine similarity, print the top-3 results with scores.

## Setup

In [ ]:
import math
import ollama

# Embedding model — separate from the chat model (Lesson 2 concept).
# Pull once with: ollama pull nomic-embed-text
EMBED_MODEL = "nomic-embed-text"

## Corpus — 12 Short Notes on Varied Topics

In [ ]:
# Twelve plain-text notes covering deliberately varied topics so the
# semantic search has clear signal to distinguish between.
NOTES = [
    "Python was created by Guido van Rossum and first released in 1991. It emphasises readability and uses significant indentation.",
    "Machine learning is a branch of AI that lets systems learn patterns from data without being explicitly programmed for each task.",
    "The Eiffel Tower is a wrought-iron lattice tower in Paris, France, built in 1889. It stands 330 metres tall including its antenna.",
    "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.",
    "A neural network is a computational model loosely inspired by the brain, made of layers of interconnected nodes called neurons.",
    "The Mediterranean diet emphasises olive oil, legumes, whole grains, fish, and vegetables, and is associated with heart health.",
    "Git is a distributed version control system created by Linus Torvalds in 2005 to manage source code history.",
    "The speed of light in a vacuum is approximately 299,792 kilometres per second, denoted by the constant c.",
    "Cosine similarity measures the angle between two vectors and returns a value between -1 and 1, where 1 means identical direction.",
    "The Amazon rainforest covers over 5.5 million square kilometres and produces roughly 20 percent of the world's oxygen.",
    "Gradient descent is an optimisation algorithm that iteratively adjusts model parameters to minimise a loss function.",
    "The Python requests library simplifies making HTTP calls: pass a URL to requests.get() and read .text or .json() on the response.",
]

print(f"Corpus: {len(NOTES)} notes")

## Concept 1 + 2 — Embedding Helper

`ollama.embeddings(model, prompt)` returns a plain Python `list[float]`.
Deterministic: same text always gives the same vector.

In [ ]:
def embed(text: str) -> list[float]:
    """Return the embedding vector for text using the configured EMBED_MODEL.

    Args:
        text: The input string to embed.

    Returns:
        A list of floats (768 values for nomic-embed-text).
    """
    # Lesson 2: ollama.embeddings() is a separate function from ollama.chat().
    # The vector lives at response["embedding"] as a plain list[float].
    response = ollama.embeddings(model=EMBED_MODEL, prompt=text)
    return response["embedding"]


# Verify the call works and returns a 768-element vector
sample_vec = embed("quick check")
print(f"Embedding model : {EMBED_MODEL}")
print(f"Vector length   : {len(sample_vec)}")   # 768
print(f"Element type    : {type(sample_vec[0]).__name__}")  # float

# Demonstrate determinism (Lesson 2): same text -> identical vector
vec_a = embed("determinism test")
vec_b = embed("determinism test")
print(f"Determinism check (same text == same vector): {vec_a == vec_b}")

## Concept 3 — Cosine Similarity

Dot product divided by the product of magnitudes. Only stdlib `math` needed.

In [ ]:
def dot_product(a: list[float], b: list[float]) -> float:
    """Sum of element-wise products of two equal-length vectors."""
    return sum(ai * bi for ai, bi in zip(a, b))


def magnitude(v: list[float]) -> float:
    """Euclidean length of vector v."""
    return math.sqrt(sum(x * x for x in v))


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two non-zero vectors.

    Args:
        a: First embedding vector.
        b: Second embedding vector, same length as a.

    Returns:
        Float in [-1, 1]. Returns 0.0 if either vector has zero magnitude.
    """
    # Guard against zero-magnitude vectors before dividing (Lesson 3).
    mag_a = magnitude(a)
    mag_b = magnitude(b)
    if mag_a == 0.0 or mag_b == 0.0:
        return 0.0
    return dot_product(a, b) / (mag_a * mag_b)


# Sanity check with toy vectors (exact answers known)
v1 = [1.0, 0.0, 0.0]
v2 = [1.0, 0.0, 0.0]
v3 = [0.0, 1.0, 0.0]
v4 = [-1.0, 0.0, 0.0]

print(f"Identical vectors    : {cosine_similarity(v1, v2):.1f}")   # 1.0
print(f"Orthogonal vectors   : {cosine_similarity(v1, v3):.1f}")   # 0.0
print(f"Opposite vectors     : {cosine_similarity(v1, v4):.1f}")   # -1.0

## Concept 4 — Build the Semantic Index

Pre-compute embeddings for all notes once at index time. At query time only
the query is embedded — documents are never re-embedded.

In [ ]:
def build_index(texts: list[str]) -> list[dict]:
    """Embed each text and return a list of index entries.

    Args:
        texts: The document strings to index.

    Returns:
        A list of dicts, each with keys 'text' (str) and 'embedding' (list[float]).
    """
    # Lesson 4: this is the expensive phase — one model call per document.
    # Run once; reuse the returned list for all subsequent queries.
    index = []
    for text in texts:
        vector = embed(text)
        index.append({"text": text, "embedding": vector})
    return index


print("Building index (this embeds every note once)...")
index = build_index(NOTES)

print(f"Index built: {len(index)} entries")
print(f"Entry keys : {list(index[0].keys())}")
print(f"Vector dim : {len(index[0]['embedding'])}")

## Concept 5 — Query-Time Search

Embed the query once, score every entry by cosine similarity, sort descending,
return the top-k results. O(n) in corpus size.

In [ ]:
def search(index: list[dict], query: str, top_k: int = 3) -> list[dict]:
    """Search the index for the most relevant entries to query.

    Args:
        index:  Pre-built index from build_index().
        query:  The natural-language search query.
        top_k:  Number of top results to return.

    Returns:
        A list of dicts with keys 'text' and 'score', sorted by score descending.
    """
    # Lesson 5: embed ONLY the query — never the indexed documents.
    query_vec = embed(query)

    # Score every entry against the query vector
    scored = [
        {
            "text": entry["text"],
            "score": cosine_similarity(query_vec, entry["embedding"]),
        }
        for entry in index
    ]

    # Sort descending by score; slice to top-k
    scored.sort(key=lambda e: e["score"], reverse=True)
    return scored[:top_k]

## Run the Search

Change `QUERY` to any string and re-run to see different results.

In [ ]:
# Change this query to explore different parts of the corpus
QUERY = "How do computers learn from data?"

results = search(index, QUERY, top_k=3)

print(f"Query: {QUERY!r}")
print("-" * 60)
for rank, result in enumerate(results, start=1):
    # Truncate long notes for display readability
    preview = result["text"] if len(result["text"]) <= 90 else result["text"][:87] + "..."
    print(f"#{rank}  score={result['score']:.4f}  {preview}")

In [ ]:
# Try a second query to show the index is reused without re-embedding documents
QUERY2 = "What is the Eiffel Tower?"

results2 = search(index, QUERY2, top_k=3)

print(f"Query: {QUERY2!r}")
print("-" * 60)
for rank, result in enumerate(results2, start=1):
    preview = result["text"] if len(result["text"]) <= 90 else result["text"][:87] + "..."
    print(f"#{rank}  score={result['score']:.4f}  {preview}")

## Deliverable Confirmation

In [ ]:
# Verify the deliverable: results exist, are sorted, and have the right shape.

assert len(results) == 3, f"expected 3 results, got {len(results)}"
assert all("score" in r and "text" in r for r in results), \
    "each result must have 'score' and 'text' keys"

scores = [r["score"] for r in results]
assert scores == sorted(scores, reverse=True), \
    "results are not sorted by score descending"

# Every score is a float in the valid cosine similarity range
assert all(-1.0 <= s <= 1.0 for s in scores), \
    "scores outside [-1, 1] — cosine similarity is broken"

# The index was built from the full corpus
assert len(index) == len(NOTES), \
    f"index has {len(index)} entries but corpus has {len(NOTES)} notes"

print("Deliverable confirmed.")
print()
print("You built a semantic search system over a corpus of notes using:")
print("  - ollama.embeddings() for free, local, deterministic text embeddings")
print("  - Cosine similarity (stdlib math only) to measure semantic closeness")
print("  - A pre-computed index (documents embedded once; queries embedded per search)")
print("  - Query-time ranking: embed → score → sort → top-k")
print()
print(f"  Corpus size : {len(NOTES)} notes")
print(f"  Vector dim  : {len(index[0]['embedding'])} floats per note (nomic-embed-text)")
print(f"  Query       : {QUERY!r}")
print(f"  Top result  : score={scores[0]:.4f}")